# H-01: Confidence Intervals for Continuous Metrics

**Question:** How fast does this agent actually respond — and how confident are we?

| | |
|---|---|
| **H₀ (Null):** | The observed IQM is a noisy estimate — the true typical performance is unknown. |
| **Hₐ (Alternative):** | The Bootstrap CI provides a reliable bound on the true typical performance. |
| **Certification rule:** | If the CI is narrow enough to be actionable, the estimate is trustworthy. |
| **Metrics:** | time_to_detect, time_to_mitigate, reasoning_score, hallucination_score |
| **Methods:** | IQM (25% trimmed mean) + Bootstrap BCa CI (10,000 resamples) |

### Aggregation Strategy
- **Per sub-fault:** IQM computed independently for each fault type (container-kill, pod-delete, etc.)
- **Category level:** Equal-weight average of sub-fault IQMs (prevents dominant sub-faults from biasing)
- **Bootstrap CI:** Resamples within each sub-fault, averages their IQMs, repeats 10k times

This ensures every fault type contributes equally to the category assessment, regardless of detection count.


In [1]:
import sys, os
import json
import numpy as np
from pathlib import Path
from collections import defaultdict

project_root = Path(os.getcwd()).parent.parent.parent
sys.path.insert(0, str(project_root))

data_root = project_root / "hypothesis_framework" / "data" / "input"

def load_runs(category_dir):
    runs = []
    for f in sorted(category_dir.glob("*.json")):
        run = json.loads(f.read_text(encoding="utf-8"))
        runs.append(run)
    return runs

def build_subfault_data(runs, metric_name, detected_only=True, source="quantitative"):
    """Group metric values by sub-fault name.

    Returns: {sub_fault_name: [values]}
    """
    grouped = defaultdict(list)
    for run in runs:
        q = run["quantitative"]
        if detected_only and q.get("fault_detected") != "Yes":
            continue
        fname = run["fault_name"]
        if source == "quantitative":
            val = q.get(metric_name)
        else:
            val = run["qualitative"].get(metric_name)
        if val is not None:
            grouped[fname].append(float(val))
    return dict(grouped)

# Load all runs per category
categories = {}
for cat_name in ["application_fault", "network_fault", "resource_fault"]:
    cat_dir = data_root / cat_name
    if cat_dir.exists():
        categories[cat_name] = load_runs(cat_dir)

print("=== Data Loaded ===")
for cat, runs in categories.items():
    total = len(runs)
    detected = sum(1 for r in runs if r["quantitative"].get("fault_detected") == "Yes")
    faults = set(r["fault_name"] for r in runs)
    print(f"  {cat}: {total} runs, {detected} detected ({detected/total:.0%})")
    for fn in sorted(faults):
        fn_runs = [r for r in runs if r["fault_name"] == fn]
        fn_det = sum(1 for r in fn_runs if r["quantitative"].get("fault_detected") == "Yes")
        print(f"    {fn}: {len(fn_runs)} runs, {fn_det} detected")


=== Data Loaded ===
  application_fault: 60 runs, 51 detected (85%)
    container-kill: 30 runs, 28 detected
    pod-delete: 30 runs, 23 detected
  network_fault: 90 runs, 39 detected (43%)
    pod-dns-error: 30 runs, 13 detected
    pod-network-corruption: 30 runs, 14 detected
    pod-network-loss: 30 runs, 12 detected
  resource_fault: 90 runs, 70 detected (78%)
    disk-fill: 30 runs, 25 detected
    pod-cpu-hog: 30 runs, 25 detected
    pod-memory-hog: 30 runs, 20 detected


---
## Step 1: Per Sub-Fault TTD Distribution

Shows why pooling is problematic — sub-faults within a category can have very different characteristics.


In [2]:
from scipy.stats import trim_mean

print("time_to_detect per sub-fault (detected-only):")
print("=" * 75)

for cat, runs in categories.items():
    subfault_data = build_subfault_data(runs, "time_to_detect")
    print(f"\n{cat}:")
    iqms = []
    for fname, vals in sorted(subfault_data.items()):
        arr = np.array(vals)
        iqm = trim_mean(arr, 0.25) if len(arr) >= 4 else np.mean(arr)
        iqms.append(iqm)
        print(f"  {fname:25s}: n={len(arr):3d}  IQM={iqm:7.1f}s  median={np.median(arr):7.1f}s  mean={np.mean(arr):7.1f}s  std={np.std(arr):6.1f}s")

    # Show pooled vs equal-weight
    all_vals = [v for vals in subfault_data.values() for v in vals]
    pooled_iqm = trim_mean(all_vals, 0.25)
    equal_weight_iqm = np.mean(iqms)
    delta = abs(pooled_iqm - equal_weight_iqm)
    print(f"  {'POOLED IQM':25s}: {pooled_iqm:7.1f}s")
    print(f"  {'EQUAL-WEIGHT IQM':25s}: {equal_weight_iqm:7.1f}s  (delta={delta:.1f}s)")
    if delta > 10:
        print(f"  NOTE: {delta:.0f}s gap — pooling would bias the category estimate")


time_to_detect per sub-fault (detected-only):

application_fault:
  container-kill           : n= 28  IQM=  122.9s  median=  120.1s  mean=  123.4s  std=  24.6s
  pod-delete               : n= 23  IQM=  131.2s  median=  130.9s  mean=  132.6s  std=  36.3s
  POOLED IQM               :   125.2s
  EQUAL-WEIGHT IQM         :   127.0s  (delta=1.9s)

network_fault:
  pod-dns-error            : n= 13  IQM=   70.0s  median=   60.0s  mean=  190.6s  std= 249.3s
  pod-network-corruption   : n= 14  IQM=  215.4s  median=  136.3s  mean=  310.5s  std= 311.9s
  pod-network-loss         : n= 12  IQM=   86.7s  median=   77.0s  mean=  190.7s  std= 206.2s
  POOLED IQM               :   120.6s
  EQUAL-WEIGHT IQM         :   124.0s  (delta=3.5s)

resource_fault:
  disk-fill                : n= 25  IQM=  223.4s  median=  226.7s  mean=  216.9s  std=  44.2s
  pod-cpu-hog              : n= 25  IQM=  235.5s  median=  262.7s  mean=  234.7s  std=  65.4s
  pod-memory-hog           : n= 20  IQM=  290.7s  median=  289.

---
## Step 2: Run H-01 — time_to_detect

Category IQM = equal-weight average of sub-fault IQMs.
Bootstrap CI resamples within each sub-fault independently, then averages.


In [3]:
from hypothesis_framework.scripts.hypothesis_tests.h01_confidence_intervals import run_confidence_interval_test

# Build input: {category: {sub_fault: [values]}}
ttd_data = {}
for cat, runs in categories.items():
    ttd_data[cat] = build_subfault_data(runs, "time_to_detect")

h01_ttd = run_confidence_interval_test(
    ttd_data,
    metric_name="time_to_detect",
    n_resamples=10000,
    random_state=42,
)

print(f"H-01: {h01_ttd.hypothesis_name}")
print(f"Metric: {h01_ttd.metric_name}")
print(f"Overall: {h01_ttd.overall_assessment}")
print("=" * 85)
for c in h01_ttd.per_category:
    width_label = "tight" if c.ci_width < 50 else "moderate" if c.ci_width < 150 else "very wide"
    print(f"\n  {c.category} (n={c.n}, {c.n_sub_faults} sub-faults):")
    print(f"    Category IQM = {c.iqm:.1f}s   95% BCa CI: [{c.ci_lower:.1f}s, {c.ci_upper:.1f}s]   width: {c.ci_width:.1f}s  ({width_label})")
    print(f"    median = {c.median:.1f}s   mean = {c.mean:.1f}s   P95 = {c.p95:.1f}s")
    print(f"    Worst sub-fault: {c.worst_sub_fault}")
    print(f"    Aggregation: {c.aggregation_method}")
    print(f"    Sub-fault breakdown:")
    for sf in c.sub_faults:
        print(f"      {sf.fault_name:25s}: IQM={sf.iqm:7.1f}s  median={sf.median:7.1f}s  n={sf.n}")

if h01_ttd.warnings:
    print(f"\nWarnings: {h01_ttd.warnings}")


H-01: Confidence Intervals for Continuous Metrics
Metric: time_to_detect
Overall: wide_intervals

  application_fault (n=51, 2 sub-faults):
    Category IQM = 127.0s   95% BCa CI: [117.2s, 137.5s]   width: 20.3s  (tight)
    median = 123.5s   mean = 127.5s   P95 = 188.5s
    Worst sub-fault: pod-delete
    Aggregation: equal_weight_subfault_iqm
    Sub-fault breakdown:
      container-kill           : IQM=  122.9s  median=  120.1s  n=28
      pod-delete               : IQM=  131.2s  median=  130.9s  n=23

  network_fault (n=39, 3 sub-faults):
    Category IQM = 124.0s   95% BCa CI: [78.6s, 270.8s]   width: 192.2s  (very wide)
    median = 80.2s   mean = 233.7s   P95 = 797.5s
    Worst sub-fault: pod-network-corruption
    Aggregation: equal_weight_subfault_iqm
    Sub-fault breakdown:
      pod-dns-error            : IQM=   70.0s  median=   60.0s  n=13
      pod-network-corruption   : IQM=  215.4s  median=  136.3s  n=14
      pod-network-loss         : IQM=   86.7s  median=   77.0s  n=

---
## Step 3: Run H-01 — time_to_mitigate


In [4]:
ttm_data = {}
for cat, runs in categories.items():
    ttm_data[cat] = build_subfault_data(runs, "time_to_mitigate")

h01_ttm = run_confidence_interval_test(ttm_data, metric_name="time_to_mitigate", n_resamples=10000, random_state=42)

print(f"H-01: {h01_ttm.metric_name}")
print("=" * 85)
for c in h01_ttm.per_category:
    width_label = "tight" if c.ci_width < 80 else "moderate" if c.ci_width < 200 else "very wide"
    print(f"  {c.category} (n={c.n}, {c.n_sub_faults} sub-faults):")
    print(f"    Category IQM = {c.iqm:.1f}s   95% CI: [{c.ci_lower:.1f}s, {c.ci_upper:.1f}s]   width: {c.ci_width:.1f}s  ({width_label})")
    for sf in c.sub_faults:
        print(f"      {sf.fault_name:25s}: IQM={sf.iqm:7.1f}s  n={sf.n}")
    print()


H-01: time_to_mitigate
  application_fault (n=51, 2 sub-faults):
    Category IQM = 313.4s   95% CI: [293.2s, 333.2s]   width: 40.0s  (tight)
      container-kill           : IQM=  300.2s  n=28
      pod-delete               : IQM=  326.7s  n=23

  network_fault (n=39, 3 sub-faults):
    Category IQM = 439.7s   95% CI: [350.2s, 590.2s]   width: 240.0s  (very wide)
      pod-dns-error            : IQM=  325.4s  n=13
      pod-network-corruption   : IQM=  530.8s  n=14
      pod-network-loss         : IQM=  462.9s  n=12

  resource_fault (n=70, 3 sub-faults):
    Category IQM = 495.4s   95% CI: [460.8s, 545.9s]   width: 85.0s  (moderate)
      pod-cpu-hog              : IQM=  479.2s  n=25
      pod-memory-hog           : IQM=  569.0s  n=20
      disk-fill                : IQM=  438.1s  n=25



---
## Step 4: Run H-01 — reasoning_quality_score


In [5]:
reasoning_data = {}
for cat, runs in categories.items():
    reasoning_data[cat] = build_subfault_data(runs, "reasoning_quality_score", source="qualitative")

h01_reasoning = run_confidence_interval_test(reasoning_data, metric_name="reasoning_quality_score", n_resamples=10000, random_state=42)

print(f"H-01: {h01_reasoning.metric_name}")
print("=" * 85)
for c in h01_reasoning.per_category:
    print(f"  {c.category} (n={c.n}):")
    print(f"    Category IQM = {c.iqm:.1f}/10   95% CI: [{c.ci_lower:.1f}, {c.ci_upper:.1f}]   width: {c.ci_width:.2f}")
    for sf in c.sub_faults:
        print(f"      {sf.fault_name:25s}: IQM={sf.iqm:.1f}/10  n={sf.n}")
    print()


H-01: reasoning_quality_score
  application_fault (n=51):
    Category IQM = 8.6/10   95% CI: [8.3, 8.8]   width: 0.55
      container-kill           : IQM=8.6/10  n=28
      pod-delete               : IQM=8.5/10  n=23

  network_fault (n=39):
    Category IQM = 5.1/10   95% CI: [4.7, 5.6]   width: 0.94
      pod-dns-error            : IQM=5.7/10  n=13
      pod-network-corruption   : IQM=5.5/10  n=14
      pod-network-loss         : IQM=4.1/10  n=12

  resource_fault (n=70):
    Category IQM = 7.1/10   95% CI: [6.8, 7.4]   width: 0.58
      pod-cpu-hog              : IQM=7.0/10  n=25
      pod-memory-hog           : IQM=7.2/10  n=20
      disk-fill                : IQM=7.2/10  n=25



---
## Step 5: Run H-01 — hallucination_score


In [6]:
hallucination_data = {}
for cat, runs in categories.items():
    hallucination_data[cat] = build_subfault_data(runs, "hallucination_score", source="qualitative")

h01_hallucination = run_confidence_interval_test(hallucination_data, metric_name="hallucination_score", n_resamples=10000, random_state=42)

print(f"H-01: {h01_hallucination.metric_name}")
print("=" * 85)
for c in h01_hallucination.per_category:
    print(f"  {c.category} (n={c.n}):")
    print(f"    Category IQM = {c.iqm:.3f}   95% CI: [{c.ci_lower:.3f}, {c.ci_upper:.3f}]   width: {c.ci_width:.4f}")
    for sf in c.sub_faults:
        print(f"      {sf.fault_name:25s}: IQM={sf.iqm:.3f}  n={sf.n}")
    print()


H-01: hallucination_score
  application_fault (n=51):
    Category IQM = 0.050   95% CI: [0.041, 0.062]   width: 0.0212
      container-kill           : IQM=0.050  n=28
      pod-delete               : IQM=0.050  n=23

  network_fault (n=39):
    Category IQM = 0.120   95% CI: [0.088, 0.151]   width: 0.0628
      pod-dns-error            : IQM=0.130  n=13
      pod-network-corruption   : IQM=0.130  n=14
      pod-network-loss         : IQM=0.100  n=12

  resource_fault (n=70):
    Category IQM = 0.080   95% CI: [0.063, 0.100]   width: 0.0370
      pod-cpu-hog              : IQM=0.090  n=25
      pod-memory-hog           : IQM=0.070  n=20
      disk-fill                : IQM=0.090  n=25



---
## Step 6: Consolidated Summary

Format per doc: `metric: IQM=Xs [lo-hi BCa CI], median=Xs, P95=Xs, mean=Xs`


In [7]:
print("H-01 Confidence Intervals — All Metrics (Equal-Weight Sub-Fault Aggregation)")
print("=" * 90)
for label, result in [("time_to_detect", h01_ttd),
                       ("time_to_mitigate", h01_ttm),
                       ("reasoning_score", h01_reasoning),
                       ("hallucination_score", h01_hallucination)]:
    print(f"\n{label}:")
    for c in result.per_category:
        unit = "s" if "time" in label else "/10" if "reasoning" in label else ""
        fmt = ".1f" if "time" in label or "reasoning" in label else ".3f"
        print(f"  {c.category}: "
              f"IQM={c.iqm:{fmt}}{unit} "
              f"[{c.ci_lower:{fmt}}-{c.ci_upper:{fmt}} BCa CI], "
              f"median={c.median:{fmt}}{unit}, "
              f"P95={c.p95:{fmt}}{unit}, "
              f"mean={c.mean:{fmt}}{unit}")


H-01 Confidence Intervals — All Metrics (Equal-Weight Sub-Fault Aggregation)

time_to_detect:
  application_fault: IQM=127.0s [117.2-137.5 BCa CI], median=123.5s, P95=188.5s, mean=127.5s
  network_fault: IQM=124.0s [78.6-270.8 BCa CI], median=80.2s, P95=797.5s, mean=233.7s
  resource_fault: IQM=249.9s [229.9-265.8 BCa CI], median=250.4s, P95=354.4s, mean=241.3s

time_to_mitigate:
  application_fault: IQM=313.4s [293.2-333.2 BCa CI], median=312.3s, P95=425.8s, mean=314.0s
  network_fault: IQM=439.7s [350.2-590.2 BCa CI], median=371.6s, P95=1090.8s, mean=511.1s
  resource_fault: IQM=495.4s [460.8-545.9 BCa CI], median=497.9s, P95=809.5s, mean=502.9s

reasoning_score:
  application_fault: IQM=8.6/10 [8.3-8.8 BCa CI], median=8.5/10, P95=10.0/10, mean=8.6/10
  network_fault: IQM=5.1/10 [4.7-5.6 BCa CI], median=5.5/10, P95=7.4/10, mean=5.2/10
  resource_fault: IQM=7.1/10 [6.8-7.4 BCa CI], median=7.1/10, P95=8.9/10, mean=7.1/10

hallucination_score:
  application_fault: IQM=0.050 [0.041-0.062

---
## Step 7: JSON Output (pipeline integration)


In [8]:
import json

output = {
    "hypothesis_id": h01_ttd.hypothesis_id,
    "hypothesis_name": h01_ttd.hypothesis_name,
    "null_hypothesis": h01_ttd.null_hypothesis,
    "alt_hypothesis": h01_ttd.alt_hypothesis,
    "certification_rule": "CI must be narrow enough to be actionable. Narrow (<50s for TTD) = trustworthy. Wide (>150s) = unreliable.",
    "hypothesis_metadata": {
        "metric_name": h01_ttd.metric_name,
        "alpha": h01_ttd.alpha,
        "confidence_level": f"{(1 - h01_ttd.alpha) * 100:.0f}%",
        "n_resamples": 10000,
        "bootstrap_method": "BCa",
        "central_tendency": "IQM (25% trimmed mean)",
        "aggregation_method": "equal_weight_subfault_iqm",
        "reference": "Agarwal et al. (NeurIPS 2021), Efron (1987)",
    },
    "overall_assessment": h01_ttd.overall_assessment,
    "confidence_intervals": {},
}

for c in h01_ttd.per_category:
    output["confidence_intervals"][c.category] = {
        "iqm": c.iqm,
        "ci_95": [c.ci_lower, c.ci_upper],
        "ci_width": c.ci_width,
        "median": c.median,
        "mean": c.mean,
        "p95": c.p95,
        "n": c.n,
        "n_sub_faults": c.n_sub_faults,
        "worst_sub_fault": c.worst_sub_fault,
        "sub_faults": {
            sf.fault_name: {
                "iqm": sf.iqm,
                "median": sf.median,
                "mean": sf.mean,
                "p95": sf.p95,
                "n": sf.n,
            }
            for sf in c.sub_faults
        },
    }

if h01_ttd.warnings:
    output["warnings"] = h01_ttd.warnings

print(json.dumps(output, indent=2))


{
  "hypothesis_id": "H-01",
  "hypothesis_name": "Confidence Intervals for Continuous Metrics",
  "null_hypothesis": "The observed IQM is a noisy estimate; the true typical performance is unknown.",
  "alt_hypothesis": "The Bootstrap CI provides a reliable bound on the true typical performance.",
  "certification_rule": "CI must be narrow enough to be actionable. Narrow (<50s for TTD) = trustworthy. Wide (>150s) = unreliable.",
  "hypothesis_metadata": {
    "metric_name": "time_to_detect",
    "alpha": 0.05,
    "confidence_level": "95%",
    "n_resamples": 10000,
    "bootstrap_method": "BCa",
    "central_tendency": "IQM (25% trimmed mean)",
    "aggregation_method": "equal_weight_subfault_iqm",
    "reference": "Agarwal et al. (NeurIPS 2021), Efron (1987)"
  },
  "overall_assessment": "wide_intervals",
  "confidence_intervals": {
    "application_fault": {
      "iqm": 127.04,
      "ci_95": [
        117.184,
        137.509
      ],
      "ci_width": 20.325,
      "median": 123.